<a href="https://colab.research.google.com/github/cncPomper/MMC/blob/master/MMC_lab_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import random


class RNG:
    """
    Klasa generatora liczb pseudolosowych bazująca na wbudowanym module random.
    Domyślnym algorytmem jest również Mersenne Twister (MT19937) w Pythonie.
    """

    def __init__(self, bok_siatki: int, seed: int):
        # Ustawiamy ziarno (seed) identyczne jak w kodzie C++ dla powtarzalności wyników
        random.seed(seed)
        self.bok_siatki = bok_siatki

    def losuj_wspolrzedna(self) -> int:
        # random.randint(a, b) losuje domknięty przedział [a, b]
        return random.randint(0, self.bok_siatki - 1)

    def losuj_z_zakresu_0_1(self) -> float:
        # random.random() zwraca liczbę z przedziału [0.0, 1.0)
        return random.random()


class ModelIsinga:

    def __init__(self, rozmiar: int, energia: int, seed: int = 123876):
        self.L = rozmiar
        self.energia_docelowa_ukladu = energia
        self.generator = RNG(rozmiar, seed)

        # Inicjalizacja pustych pól klasowych (odpowiedniki pól z nagłówka .h)
        self.energia_poczatkowa_ukladu = 0
        self.energia_duszka = 0
        self.magnetyzacja = 0

        self.srednia_energia_ukladu = 0.0
        self.srednia_energia_duszka = 0.0
        self.srednia_magnetyzacja = 0.0
        self.temperatura = 0.0

        # Tworzenie dwuwymiarowej siatki wypełnionej zerami (alokacja pamięci)
        self.siatka = [[0 for _ in range(self.L)] for _ in range(self.L)]

    def ustaw_same_jedynki(self):
        for i in range(self.L):
            for j in range(self.L):
                self.siatka[i][j] = 1

    def deltaE(self, x: int, y: int) -> int:
        # Periodyczne warunki brzegowe
        gorny = x - 1 if x != 0 else self.L - 1
        dolny = x + 1 if x != self.L - 1 else 0
        prawy = y + 1 if y != self.L - 1 else 0
        lewy = y - 1 if y != 0 else self.L - 1

        E_pocz = self.siatka[x][y] * (
            self.siatka[x][lewy]
            + self.siatka[x][prawy]
            + self.siatka[gorny][y]
            + self.siatka[dolny][y]
        )
        E_konc = -E_pocz

        return E_pocz - E_konc

    def sprobuj_odwrocic_spin_losowego_atomu(self):
        i = self.generator.losuj_wspolrzedna()
        j = self.generator.losuj_wspolrzedna()
        dE = self.deltaE(i, j)

        if dE <= self.energia_duszka:
            self.siatka[i][j] = -self.siatka[i][j]
            self.energia_duszka -= dE
            self.energia_poczatkowa_ukladu += dE
            self.magnetyzacja += 2 * self.siatka[i][j]

    def doprowadzenie_do_stanu_rownowagi(self, liczba_krokow: int):
        self.magnetyzacja = self.L * self.L
        self.energia_poczatkowa_ukladu = -2 * self.L * self.L
        self.energia_duszka = (
            self.energia_docelowa_ukladu - self.energia_poczatkowa_ukladu
        )
        self.ustaw_same_jedynki()

        for _ in range(liczba_krokow):
            self.sprobuj_odwrocic_spin_losowego_atomu()

    def zliczanie_srednich(self, liczba_krokow: int):
        energia_duszka_do_sredniej = 0
        energia_ukladu_do_sredniej = 0
        magnetyzacja_tot = 0

        # Pętla liczby_kroków
        for _ in range(liczba_krokow):
            # Pętla statystycznie po każdym spinie
            for _ in range(self.L * self.L):
                self.sprobuj_odwrocic_spin_losowego_atomu()

            energia_duszka_do_sredniej += self.energia_duszka
            energia_ukladu_do_sredniej += self.energia_poczatkowa_ukladu
            magnetyzacja_tot += abs(self.magnetyzacja)

        # Obliczanie średnich
        self.srednia_energia_duszka = energia_duszka_do_sredniej / liczba_krokow
        self.srednia_energia_ukladu = energia_ukladu_do_sredniej / liczba_krokow
        self.srednia_magnetyzacja = (
            magnetyzacja_tot / liczba_krokow / (self.L * self.L)
        )

        try:
            if self.srednia_energia_duszka > 0:
              self.temperatura = 4.0 / (
                  math.log(1 + 4.0 / self.srednia_energia_duszka)
              )
            else:
                self.temperatura = 0.0
        except ZeroDivisionError:
            self.temperatura = float("inf")  # Obsługa dzielenia przez zero

    def podaj_srednia_energie_duszka(self) -> float:
        return self.srednia_energia_duszka

    def podaj_srednia_energie_ukladu(self) -> float:
        return self.srednia_energia_ukladu

    def podaj_srednia_magnetyzacje(self) -> float:
        return self.srednia_magnetyzacja

    def podaj_temperature(self) -> float:
        return self.temperatura

In [ ]:
ising = ModelIsinga(10, -184, 20392)
ising.doprowadzenie_do_stanu_rownowagi(1000)
ising.zliczanie_srednich(1000)

print(f"Srednia energia ukladu = {ising.podaj_srednia_energie_ukladu()}")
print(f"Srednia energia duszka = {ising.podaj_srednia_energie_duszka()}")
print(f"Srednia magnetyzacja   = {ising.podaj_srednia_magnetyzacje()}")
print(f"Temperatura            = {ising.podaj_temperature()}")

In [ ]:
print("Symulacja modelu Isinga w Zespole Mikrokanonicznym (Zadanie 1)")
print("-" * 75)
print(
    f"{'L':<5} | {'E_docelowa':<12} | {'<E_ukladu>':<12} | {'<E_duszka>':<12} | {'<m>':<10} | {'T':<10}"
)
print("-" * 75)

konfiguracja_zadan = [
        {"L": 10, "E_min": -184, "E_max": -24, "krok": 8},
        {"L": 20, "E_min": -768, "E_max": -32, "krok": 32},
        {"L": 40, "E_min": -3072, "E_max": -128, "krok": 128},
    ]

KROKI_ROWNOWAGI = 2000
KROKI_SREDNICH = 2000

for konfig in konfiguracja_zadan:
    L = konfig["L"]
    E_aktualna = konfig["E_min"]

    # Iteracja po zadanym przedziale energii domkniętej [E_min, E_max]
    while E_aktualna <= konfig["E_max"]:
        ising = ModelIsinga(L, E_aktualna)

        # 1. Doprowadzenie do stanu równowagi
        ising.doprowadzenie_do_stanu_rownowagi(KROKI_ROWNOWAGI)

        # 2. Zliczanie średnich statystycznych
        ising.zliczanie_srednich(KROKI_SREDNICH)

        # Wypisanie sformatowanych wyników dla bieżącego punktu pomiarowego
        print(
            f"{L:<5} | {E_aktualna:<12} | "
            f"{ising.podaj_srednia_energie_ukladu():<12.4f} | "
            f"{ising.podaj_srednia_energie_duszka():<12.4f} | "
            f"{ising.podaj_srednia_magnetyzacje():<10.4f} | "
            f"{ising.podaj_temperature():<10.4f}"
        )

        E_aktualna += konfig["krok"]
    print("-" * 75)

In [4]:
# ==============================================================================
# KLASA POLIMORFICZNA DLA ZADANIA 2
# ==============================================================================


class ModelIsingaKanoniczny(ModelIsinga):
    """Klasa pochodna - reprezentuje układ kanoniczny (Zadanie 2).

    Dziedziczy mechanikę sieci i średnich, redefiniuje logikę Metropolis.
    """

    def __init__(self, rozmiar: int, temperatura: float, seed: int = 11100272):
        # Wywołujemy konstruktor rodzica przekazując 0 jako energię docelową
        # (w modelu kanonicznym nie jest ona używana)
        super().__init__(rozmiar, energia=0, seed=seed)
        self.T_zredukowana = temperatura

    def sprobuj_odwrocic_spin_losowego_atomu(self):
        """Nadpisanie (Override) metody: Kryterium Metropolis (kanoniczne)."""
        i = self.generator.losuj_wspolrzedna()
        j = self.generator.losuj_wspolrzedna()
        dE = self.deltaE(i, j)

        if dE <= 0:
            akceptacja = True
        else:
            if self.T_zredukowana > 0:
                prawdopodobienstwo = math.exp(-dE / self.T_zredukowana)
                akceptacja = self.generator.losuj_z_zakresu_0_1() < prawdopodobienstwo
            else:
                akceptacja = False

        if akceptacja:
            self.siatka[i][j] = -self.siatka[i][j]
            self.energia_poczatkowa_ukladu += dE  # Korzystamy z pola rodzica
            self.magnetyzacja += 2 * self.siatka[i][j]

    def doprowadzenie_do_stanu_rownowagi(self, liczba_krokow: int):
        """Dostosowanie przygotowania układu (brak duszka)."""
        self.magnetyzacja = self.L * self.L
        self.energia_poczatkowa_ukladu = -2 * self.L * self.L
        self.ustaw_same_jedynki()

        for _ in range(liczba_krokow):
            self.sprobuj_odwrocic_spin_losowego_atomu()

In [ ]:
print("Test dziedziczenia: Uruchomienie układu kanonicznego (Metropolis)")
print("-" * 55)
print(f"{'L':<5} | {'T_zred':<12} | {'<E_ukladu>':<14} | {'<m>':<10}")
print("-" * 55)

# Przykład dla siatki L=10 i kilku temperatur
L = 10
for T in [1.5, 2.0, 2.269, 3.0]:
    model_kanoniczny = ModelIsingaKanoniczny(L, T)
    model_kanoniczny.doprowadzenie_do_stanu_rownowagi(2000)
    model_kanoniczny.zliczanie_srednich(2000)

    print(
        f"{L:<5} | {T:<12.3f} | "
        f"{model_kanoniczny.podaj_srednia_energie_ukladu():<14.4f} | "
        f"{model_kanoniczny.podaj_srednia_magnetyzacje():<10.4f}"
    )